# Eksperimen Robustness: Ketidakpastian, Threshold, SHAP, dan Validasi Lintas Sumber

**Tujuan:** menguji interval ketidakpastian, perbandingan DDS/noDDS berpasangan, threshold yang dipilih tanpa melihat test, kontribusi SHAP dengan satu metode, dan uji pemisahan sumber LIN-ke-PBN dalam dataset gabungan.

Notebook ini membaca artefak eksperimen tersimpan. Untuk mengulang seluruh eksperimen (bootstrap 500 replikasi, SHAP, serta fit enam model untuk evaluasi source-held-out internal pada data gabungan), jalankan `run_robustness.py` dari root repo. Skrip hanya menulis ke folder `results/` eksperimen ini.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'configs' / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / 'experiments' / '2026-10-21' / 'results'
assert OUT.exists(), f'Hasil belum dibuat: {OUT}. Jalankan run_robustness.py.'
print('Repo:', ROOT)
print('Hasil eksperimen:', OUT)

Repo: D:\Bridge-Prediction
Hasil eksperimen: D:\Bridge-Prediction\experiments\2026-10-21\results


## 1. Interval ketidakpastian baseline

CI 95% dihitung dengan 500 bootstrap pada unit `_deal_key`; semua record dari satu deal diambil sebagai satu cluster. Ini menghindari asumsi bahwa beberapa rekaman dari deal yang sama saling independen.

In [2]:
baseline_ci = pd.read_csv(OUT / 'baseline_test_bootstrap_ci.csv')
cols = ['dataset','algorithm','accuracy','accuracy_ci95_low','accuracy_ci95_high',
        'f1_macro','f1_macro_ci95_low','f1_macro_ci95_high',
        'average_precision','average_precision_ci95_low','average_precision_ci95_high']
display(baseline_ci[cols].style.format({c:'{:.3f}' for c in cols if c not in ('dataset','algorithm')}))

,dataset,algorithm,accuracy,accuracy_ci95_low,accuracy_ci95_high,f1_macro,f1_macro_ci95_low,f1_macro_ci95_high,average_precision,average_precision_ci95_low,average_precision_ci95_high
0,DDS,RandomForest,0.803,0.793,0.815,0.710,0.694,0.724,0.589,0.559,0.620
1,DDS,XGBoost,0.809,0.797,0.819,0.695,0.678,0.710,0.595,0.563,0.627
2,DDS,LightGBM,0.792,0.781,0.803,0.729,0.715,0.744,0.628,0.600,0.659
3,noDDS,RandomForest,0.750,0.738,0.762,0.600,0.582,0.616,0.436,0.401,0.471
4,noDDS,XGBoost,0.760,0.748,0.773,0.545,0.527,0.562,0.439,0.407,0.473
5,noDDS,LightGBM,0.714,0.702,0.727,0.637,0.621,0.653,0.418,0.387,0.451


## 2. Perbandingan berpasangan DDS dan noDDS

Selisih metrik dihitung pada board yang sama. Interval selisih memakai bootstrap cluster per deal. Uji sign eksak membandingkan jumlah deal yang kinerjanya lebih baik pada DDS dengan jumlah yang lebih baik pada noDDS; p-value dikoreksi Holm untuk tiga algoritma.

In [3]:
paired = pd.read_csv(OUT / 'paired_dds_vs_nodds_inference.csv')
paired_cols = ['algorithm','accuracy_delta_dds_minus_nodds',
 'paired_bootstrap_accuracy_delta_ci95_low','paired_bootstrap_accuracy_delta_ci95_high',
 'f1_macro_delta_dds_minus_nodds','paired_bootstrap_f1_macro_delta_ci95_low','paired_bootstrap_f1_macro_delta_ci95_high',
 'ap_delta_dds_minus_nodds','paired_bootstrap_ap_delta_ci95_low','paired_bootstrap_ap_delta_ci95_high',
 'deal_groups_dds_higher_accuracy','deal_groups_nodds_higher_accuracy','deal_groups_tied_accuracy',
 'exact_sign_test_pvalue_deal_level_accuracy','holm_adjusted_sign_test_pvalue']
display(paired[paired_cols].style.format({c:'{:.4g}' for c in paired_cols if c not in ('algorithm','deal_groups_dds_higher_accuracy','deal_groups_nodds_higher_accuracy','deal_groups_tied_accuracy')}))

,algorithm,accuracy_delta_dds_minus_nodds,paired_bootstrap_accuracy_delta_ci95_low,paired_bootstrap_accuracy_delta_ci95_high,f1_macro_delta_dds_minus_nodds,paired_bootstrap_f1_macro_delta_ci95_low,paired_bootstrap_f1_macro_delta_ci95_high,ap_delta_dds_minus_nodds,paired_bootstrap_ap_delta_ci95_low,paired_bootstrap_ap_delta_ci95_high,deal_groups_dds_higher_accuracy,deal_groups_nodds_higher_accuracy,deal_groups_tied_accuracy,exact_sign_test_pvalue_deal_level_accuracy,holm_adjusted_sign_test_pvalue
0,RandomForest,0.05324,0.0421,0.06409,0.1096,0.09406,0.1255,0.1532,0.1314,0.1719,375,146,2879,3.029e-24,6.058e-24
1,XGBoost,0.04899,0.0397,0.06021,0.1502,0.133,0.1677,0.156,0.1318,0.1813,345,129,2926,8.578e-24,8.578e-24
2,LightGBM,0.07754,0.06392,0.09075,0.09274,0.07844,0.1064,0.2105,0.1872,0.2342,555,216,2629,3.301e-35,9.902e-35


## 3. Threshold validation dan kinerja test

Threshold dipilih dari validation dengan memaksimalkan F1 macro. Tabel memperlihatkan trade-off pada test. AP dan ROC-AUC tidak berubah oleh threshold karena metrik tersebut menggunakan skor probabilitas.

In [4]:
thresholds = pd.read_csv(OUT / 'threshold_validation_test.csv')
threshold_cols = ['dataset','algorithm','validation_selected_threshold',
 'default_accuracy','validation_threshold_accuracy',
 'default_f1_macro','validation_threshold_f1_macro',
 'default_f1_positive','validation_threshold_f1_positive',
 'validation_threshold_precision_positive','validation_threshold_recall_positive',
 'validation_threshold_tn','validation_threshold_fp','validation_threshold_fn','validation_threshold_tp']
display(thresholds[threshold_cols].style.format({c:'{:.3f}' for c in threshold_cols if c not in ('dataset','algorithm')}))

,dataset,algorithm,validation_selected_threshold,default_accuracy,validation_threshold_accuracy,default_f1_macro,validation_threshold_f1_macro,default_f1_positive,validation_threshold_f1_positive,validation_threshold_precision_positive,validation_threshold_recall_positive,validation_threshold_tn,validation_threshold_fp,validation_threshold_fn,validation_threshold_tp
0,DDS,RandomForest,0.435,0.803,0.791,0.710,0.721,0.545,0.581,0.572,0.590,4869.000,814.000,759.000,1090.000
1,DDS,XGBoost,0.375,0.809,0.804,0.695,0.727,0.508,0.583,0.610,0.558,5023.000,660.000,817.000,1032.000
2,DDS,LightGBM,0.585,0.792,0.812,0.729,0.740,0.600,0.603,0.625,0.582,5038.000,645.000,773.000,1076.000
3,noDDS,RandomForest,0.435,0.750,0.728,0.600,0.637,0.355,0.456,0.448,0.464,4624.000,1059.000,991.000,858.000
4,noDDS,XGBoost,0.345,0.760,0.739,0.545,0.636,0.231,0.443,0.465,0.423,4782.000,901.000,1066.000,783.000
5,noDDS,LightGBM,0.520,0.714,0.724,0.637,0.636,0.469,0.458,0.442,0.475,4573.000,1110.000,970.000,879.000


## 4. SHAP dengan metode konsisten

TreeExplainer default dipakai pada 50 board validation yang sama untuk keenam model. Proporsi mean |SHAP| adalah ukuran kepentingan global relatif; tidak menunjukkan arah pengaruh atau sebab-akibat. Sampel 50 board menjadikan ini pemeriksaan eksploratif, sehingga kesimpulan utama tetap dikaitkan dengan evaluasi kinerja DDS/noDDS.

In [5]:
shap_results = pd.read_csv(OUT / 'matched_exact_tree_shap.csv')
display(shap_results.style.format({'dds_feature_mean_abs_shap_share':'{:.2%}'}))

,dataset,algorithm,n_validation_rows,method,dds_feature_mean_abs_shap_share,top_10_features
0,DDS,RandomForest,50,TreeExplainer default exact Tree SHAP; LightGBM native implementation,28.47%,"ew_dd_N, ns_dd_N, ew_hcp, ns_hcp_advantage, ns_hcp, ns_dd_S, ew_dd_S, ns_dd_H, ew_dd_H, ew_controls"
1,DDS,XGBoost,50,TreeExplainer default exact Tree SHAP; LightGBM native implementation,30.52%,"ns_hcp, ns_dd_N, ew_dd_N, ns_dd_S, auction_has_double, ns_dd_H, ew_dd_S, ew_dd_H, auction_ew_bids, auction_ns_bids"
2,DDS,LightGBM,50,TreeExplainer default exact Tree SHAP; LightGBM native implementation,31.55%,"ns_hcp, ns_dd_N, ns_dd_H, auction_has_double, ew_dd_N, ns_dd_S, auction_ew_bids, ew_dd_H, ew_dd_D, ew_dd_S"
3,noDDS,RandomForest,50,TreeExplainer default exact Tree SHAP; LightGBM native implementation,0.00%,"ew_hcp, ns_hcp_advantage, ns_hcp, ns_controls, ew_controls, auction_ew_bids, auction_ns_bids, auction_has_double, auction_doubled, ew_ltc"
4,noDDS,XGBoost,50,TreeExplainer default exact Tree SHAP; LightGBM native implementation,0.00%,"ns_hcp, auction_has_double, ns_controls, auction_ew_bids, auction_ns_bids, auction_competitive, ew_hcp, auction_len, ns_ltc, ew_ltc"
5,noDDS,LightGBM,50,TreeExplainer default exact Tree SHAP; LightGBM native implementation,0.00%,"ns_hcp, auction_has_double, ns_controls, auction_ew_bids, auction_ns_bids, ns_fit_D, ew_hcp, E_hcp_C, ew_ltc, ew_best_fit"


## 5. Uji source-held-out internal pada dataset gabungan LIN dan PBN

Dataset penelitian utama sudah menggabungkan LIN dan PBN. Untuk uji tambahan ini, bagian LIN dari dataset gabungan dipakai untuk melatih model dan memilih threshold melalui validation deal-aware; model kemudian dilatih ulang pada seluruh LIN dan dievaluasi pada bagian PBN yang ditahan. PBN tidak digunakan untuk fitting atau memilih threshold. Ini validasi source-held-out internal, bukan validasi eksternal independen.

In [6]:
source_eval = pd.read_csv(OUT / 'source_heldout_lin_to_pbn.csv')
source_cols = ['dataset','algorithm','n_train_lin','n_heldout_pbn',
 'lin_positive_rate','pbn_positive_rate','threshold_selected_on_lin_validation',
 'default_accuracy','default_f1_macro','default_average_precision',
 'lin_threshold_f1_positive','lin_threshold_average_precision']
display(source_eval[source_cols].style.format({c:'{:.3f}' for c in source_cols if c not in ('dataset','algorithm','n_train_lin','n_heldout_pbn')}))

,dataset,algorithm,n_train_lin,n_heldout_pbn,lin_positive_rate,pbn_positive_rate,threshold_selected_on_lin_validation,default_accuracy,default_f1_macro,default_average_precision,lin_threshold_f1_positive,lin_threshold_average_precision
0,DDS,RandomForest,12471,37756,0.243,0.246,0.415,0.793,0.675,0.564,0.552,0.564
1,DDS,XGBoost,12471,37756,0.243,0.246,0.315,0.790,0.641,0.553,0.553,0.553
2,DDS,LightGBM,12471,37756,0.243,0.246,0.450,0.793,0.711,0.573,0.564,0.573
3,noDDS,RandomForest,12471,37756,0.243,0.246,0.420,0.755,0.596,0.430,0.450,0.430
4,noDDS,XGBoost,12471,37756,0.243,0.246,0.250,0.757,0.533,0.413,0.461,0.413
5,noDDS,LightGBM,12471,37756,0.243,0.246,0.410,0.737,0.617,0.407,0.447,0.407


## 6. Catatan metodologis

- Test set dipakai untuk estimasi akhir; threshold ditetapkan dari validation.
- Bootstrap mengukur ketidakpastian sampling test pada tingkat deal, bukan variabilitas antarrun pelatihan.
- SHAP menjelaskan kontribusi fitur terhadap fungsi prediksi model; tidak mengidentifikasi hubungan sebab-akibat.
- Dataset utama sudah menggabungkan sumber LIN dan PBN. Uji source-held-out LIN-ke-PBN memeriksa transfer antarbagian sumber dari koleksi yang sama; ini bukan validasi eksternal independen.

In [7]:
meta = __import__('json').loads((OUT / 'metadata.json').read_text(encoding='utf-8'))
meta

{'created': '2026-10-21',
 'test_bootstrap_replicates': 500,
 'shap_sample_size': 50,
 'shap_method': 'TreeExplainer default exact Tree SHAP path-dependent method across all six models; LightGBM uses its native Tree SHAP implementation',
 'threshold_selection': 'maximize macro-F1 over validation probabilities, threshold grid 0.05 to 0.95 in steps of 0.005',
 'paired_test_design': 'deal-cluster bootstrap DDS minus noDDS and exact sign test on deal-level accuracy differences; test set used only for final estimation',
 'bootstrap_unit': '_deal_key groups sampled with replacement; within-deal rows remain together; class labels may differ within a deal',
 'positive_class_rate_test': 0.2454859267126925,
 'source_heldout_design': 'internal source-held-out assessment on the already merged LIN+PBN dataset: train on LIN rows only; choose threshold on grouped 20% LIN validation using deal-aware StratifiedGroupKFold; refit on all LIN; evaluate on the PBN subset',
 'source_heldout_scope': 'source-s